## CSP


In [1]:
"""
=============================================================================
VEHICLE ROUTING & BIN PACKING OPTIMIZATION
=============================================================================
Progetto: Amazon Delivery AI
Autore: Sistema di Ottimizzazione Flotta
Descrizione: Script per l'assegnazione ottimale dei pacchi a 5 furgoni
             utilizzando tecniche di Ricerca Operativa:
             - De-standardizzazione dei tempi predetti
             - Clustering geografico (K-Means)
             - Bin Packing Greedy per ottimizzazione carico

Vincoli del problema:
- 5 Furgoni disponibili
- Turno lavorativo: 8 ore (480 minuti)
- Tempo consegna: da 10 a 120 minuti per pacco
=============================================================================
"""

import pandas as pd
import numpy as np
from sklearn.cluster import KMeans
from sklearn.preprocessing import MinMaxScaler
import warnings

# Sopprimiamo warning non critici per pulizia output
warnings.filterwarnings('ignore')

# =============================================================================
# CONFIGURAZIONE PARAMETRI
# =============================================================================

# Parametri del problema
NUM_VANS = 5                    # Numero di furgoni nella flotta
VAN_CAPACITY_MINUTES = 480      # Capacità massima per furgone (8 ore)
MIN_DELIVERY_TIME = 10          # Tempo minimo consegna (minuti)
MAX_DELIVERY_TIME = 120         # Tempo massimo consegna (minuti)

# Path dei file
INPUT_FILE = '../Supervised_Learning/amazon_delivery_with_predictions.csv'
OUTPUT_FILE = 'amazon_delivery_optimized.csv'


def load_data(filepath: str) -> pd.DataFrame:
    """
    Carica il dataset delle consegne con le predizioni ML.
    
    Args:
        filepath: Percorso del file CSV
        
    Returns:
        DataFrame con i dati delle consegne
    """
    print("=" * 60)
    print("CARICAMENTO DATI")
    print("=" * 60)
    
    df = pd.read_csv(filepath)
    
    print(f"✓ File caricato: {filepath}")
    print(f"✓ Numero totale consegne: {len(df)}")
    print(f"✓ Colonne disponibili: {list(df.columns)}")
    print(f"✓ Range Predicted_Time: [{df['Predicted_Time'].min():.3f}, {df['Predicted_Time'].max():.3f}]")
    
    return df


# =============================================================================
# STEP 1: DE-STANDARDIZZAZIONE DEI TEMPI (MAPPING)
# =============================================================================

def destandardize_time(df: pd.DataFrame) -> pd.DataFrame:
    """
    STEP 1: De-Standardizzazione dei Tempi Predetti
    
    I valori in 'Predicted_Time' sono z-score (media=0, std=1).
    Non possiamo usarli direttamente per pianificare minuti reali.
    
    Tecnica utilizzata: MinMax Scaling Inverso (Mapping Lineare)
    
    Formula matematica:
        Time_Minutes = (z - z_min) / (z_max - z_min) * (max_time - min_time) + min_time
    
    Dove:
        - z: valore z-score originale
        - z_min, z_max: estremi dei z-score nel dataset
        - min_time: 10 minuti (tempo minimo realistico)
        - max_time: 120 minuti (tempo massimo realistico)
    
    Args:
        df: DataFrame con colonna 'Predicted_Time' (z-score)
        
    Returns:
        DataFrame con nuova colonna 'Time_Minutes'
    """
    print("\n" + "=" * 60)
    print("STEP 1: DE-STANDARDIZZAZIONE DEI TEMPI")
    print("=" * 60)
    
    # Estrazione valori z-score
    z_scores = df['Predicted_Time'].values
    
    # Calcolo estremi per il mapping
    z_min = z_scores.min()
    z_max = z_scores.max()
    
    print(f"\n📊 Statistiche z-score originali:")
    print(f"   • Minimo (z_min): {z_min:.4f}")
    print(f"   • Massimo (z_max): {z_max:.4f}")
    print(f"   • Media: {z_scores.mean():.4f}")
    print(f"   • Deviazione Std: {z_scores.std():.4f}")
    
    # Mapping lineare: z-score → minuti reali
    # Formula: normalized = (z - z_min) / (z_max - z_min)
    #          minutes = normalized * (max_time - min_time) + min_time
    
    if z_max - z_min == 0:
        # Caso degenere: tutti i valori uguali
        df['Time_Minutes'] = (MIN_DELIVERY_TIME + MAX_DELIVERY_TIME) // 2
    else:
        # Normalizzazione [0, 1]
        normalized = (z_scores - z_min) / (z_max - z_min)
        # Scaling all'intervallo [10, 120]
        minutes = normalized * (MAX_DELIVERY_TIME - MIN_DELIVERY_TIME) + MIN_DELIVERY_TIME
        # Arrotondamento a interi
        df['Time_Minutes'] = np.round(minutes).astype(int)
    
    print(f"\n⏱️  Mapping completato:")
    print(f"   • z-score {z_min:.2f} → {MIN_DELIVERY_TIME} minuti")
    print(f"   • z-score {z_max:.2f} → {MAX_DELIVERY_TIME} minuti")
    print(f"\n📈 Statistiche Time_Minutes:")
    print(f"   • Minimo: {df['Time_Minutes'].min()} minuti")
    print(f"   • Massimo: {df['Time_Minutes'].max()} minuti")
    print(f"   • Media: {df['Time_Minutes'].mean():.1f} minuti")
    print(f"   • Mediana: {df['Time_Minutes'].median():.1f} minuti")
    
    return df


# =============================================================================
# STEP 2: CLUSTERING GEOGRAFICO (K-MEANS - UNSUPERVISED LEARNING)
# =============================================================================

def geographic_clustering(df: pd.DataFrame) -> pd.DataFrame:
    """
    STEP 2: Clustering Geografico con K-Means
    
    Obiettivo: Dividere la città in 5 zone geografiche, una per furgone.
    Questo simula l'assegnazione territoriale ottimale basata sulla
    vicinanza geografica dei punti di consegna.
    
    Algoritmo: K-Means Clustering (Unsupervised Learning)
    
    K-Means minimizza la varianza intra-cluster:
        J = Σᵢ Σₓ∈Cᵢ ||x - μᵢ||²
    
    Dove:
        - Cᵢ: cluster i-esimo
        - μᵢ: centroide del cluster i
        - x: punto dati (coordinate lat/long)
    
    Args:
        df: DataFrame con colonne 'Drop_Latitude' e 'Drop_Longitude'
        
    Returns:
        DataFrame con nuova colonna 'Van_ID' (0-4)
    """
    print("\n" + "=" * 60)
    print("STEP 2: CLUSTERING GEOGRAFICO (K-MEANS)")
    print("=" * 60)
    
    # Estrazione coordinate geografiche
    coordinates = df[['Drop_Latitude', 'Drop_Longitude']].values
    
    print(f"\n🌍 Coordinate geografiche:")
    print(f"   • Latitudine: [{coordinates[:, 0].min():.4f}, {coordinates[:, 0].max():.4f}]")
    print(f"   • Longitudine: [{coordinates[:, 1].min():.4f}, {coordinates[:, 1].max():.4f}]")
    
    # Configurazione K-Means
    # - n_clusters=5: un cluster per ogni furgone
    # - random_state=42: riproducibilità dei risultati
    # - n_init=10: numero di inizializzazioni per evitare minimi locali
    kmeans = KMeans(
        n_clusters=NUM_VANS,
        random_state=42,
        n_init=10,
        max_iter=300
    )
    
    # Fitting del modello e predizione dei cluster
    df['Van_ID'] = kmeans.fit_predict(coordinates)
    
    # Estrazione centroidi (posizione centrale di ogni zona)
    centroids = kmeans.cluster_centers_
    
    print(f"\n🚚 Clustering completato con {NUM_VANS} zone:")
    print("-" * 50)
    
    for van_id in range(NUM_VANS):
        n_packages = (df['Van_ID'] == van_id).sum()
        centroid = centroids[van_id]
        print(f"   Furgone {van_id}:")
        print(f"      • Pacchi assegnati: {n_packages}")
        print(f"      • Centro zona: ({centroid[0]:.4f}, {centroid[1]:.4f})")
    
    # Calcolo metrica di qualità del clustering (inerzia)
    print(f"\n📊 Qualità clustering:")
    print(f"   • Inerzia (somma distanze²): {kmeans.inertia_:.2f}")
    
    return df


# =============================================================================
# STEP 3: OTTIMIZZAZIONE DEL CARICO (BIN PACKING GREEDY)
# =============================================================================

def bin_packing_optimization(df: pd.DataFrame) -> pd.DataFrame:
    """
    STEP 3: Bin Packing Greedy per Ottimizzazione Carico
    
    Problema: Bin Packing con vincolo di capacità
    
    Vincolo: Ogni furgone ha capacità massima di 480 minuti (8 ore)
    
    Strategia: Greedy "First Fit Decreasing" modificato
        1. Per ogni furgone, ordinare i pacchi dal più veloce al più lento
        2. Selezionare pacchi finché la capacità non è esaurita
        3. Questa strategia massimizza il NUMERO di consegne per turno
    
    Nota: Ordinare dal più veloce massimizza il throughput (numero consegne).
          Ordinare dal più lento massimizzerebbe l'utilizzo della capacità.
    
    Args:
        df: DataFrame con colonne 'Van_ID' e 'Time_Minutes'
        
    Returns:
        DataFrame con nuova colonna 'Status' ('Loaded' o 'Skipped')
    """
    print("\n" + "=" * 60)
    print("STEP 3: BIN PACKING OPTIMIZATION (GREEDY)")
    print("=" * 60)
    print(f"\n⚙️  Vincolo: Capacità furgone = {VAN_CAPACITY_MINUTES} minuti (8 ore)")
    print(f"📋 Strategia: Greedy - Priorità ai pacchi più veloci")
    
    # Inizializzazione colonna Status
    df['Status'] = 'Skipped'  # Default: non caricato
    
    # Dizionario per memorizzare statistiche per il report
    van_stats = {}
    
    print("\n" + "-" * 60)
    
    # Iterazione su ogni furgone
    for van_id in range(NUM_VANS):
        # Filtro pacchi della zona corrente
        van_mask = df['Van_ID'] == van_id
        van_packages = df[van_mask].copy()
        
        # Ordinamento: dal più veloce al più lento (strategia greedy)
        # Questo massimizza il numero di consegne nel turno
        van_packages_sorted = van_packages.sort_values('Time_Minutes', ascending=True)
        
        # Variabili per il bin packing
        total_time = 0
        loaded_count = 0
        loaded_indices = []
        
        # Algoritmo Greedy: First Fit
        for idx, row in van_packages_sorted.iterrows():
            delivery_time = row['Time_Minutes']
            
            # Controllo vincolo di capacità
            if total_time + delivery_time <= VAN_CAPACITY_MINUTES:
                # Il pacco può essere caricato
                total_time += delivery_time
                loaded_count += 1
                loaded_indices.append(idx)
        
        # Aggiornamento Status per i pacchi caricati
        df.loc[loaded_indices, 'Status'] = 'Loaded'
        
        # Calcolo statistiche
        total_packages = len(van_packages)
        skipped_count = total_packages - loaded_count
        utilization = (total_time / VAN_CAPACITY_MINUTES) * 100
        
        # Salvataggio statistiche per report finale
        van_stats[van_id] = {
            'total_packages': total_packages,
            'loaded': loaded_count,
            'skipped': skipped_count,
            'total_time': total_time,
            'utilization': utilization
        }
        
        # Output progressivo
        print(f"\n🚚 FURGONE {van_id}:")
        print(f"   📦 Pacchi zona: {total_packages}")
        print(f"   ✅ Caricati: {loaded_count}")
        print(f"   ❌ Esclusi: {skipped_count}")
        print(f"   ⏱️  Tempo totale: {total_time}/{VAN_CAPACITY_MINUTES} minuti")
        print(f"   📊 Utilizzo turno: {utilization:.1f}%")
    
    return df, van_stats


def print_final_report(df: pd.DataFrame, van_stats: dict):
    """
    Stampa il report finale dell'ottimizzazione.
    
    Args:
        df: DataFrame ottimizzato
        van_stats: Dizionario con statistiche per furgone
    """
    print("\n" + "=" * 60)
    print("REPORT FINALE OTTIMIZZAZIONE FLOTTA")
    print("=" * 60)
    
    # Statistiche globali
    total_packages = len(df)
    total_loaded = (df['Status'] == 'Loaded').sum()
    total_skipped = (df['Status'] == 'Skipped').sum()
    
    print(f"\n📊 STATISTICHE GLOBALI:")
    print(f"   • Pacchi totali nel dataset: {total_packages}")
    print(f"   • Pacchi caricati (Loaded): {total_loaded} ({total_loaded/total_packages*100:.1f}%)")
    print(f"   • Pacchi esclusi (Skipped): {total_skipped} ({total_skipped/total_packages*100:.1f}%)")
    
    # Tabella riepilogativa
    print(f"\n📋 RIEPILOGO PER FURGONE:")
    print("-" * 70)
    print(f"{'Furgone':<10} {'Caricati':<12} {'Tempo (min)':<15} {'Utilizzo %':<12} {'Efficienza':<12}")
    print("-" * 70)
    
    total_time_all = 0
    for van_id in range(NUM_VANS):
        stats = van_stats[van_id]
        efficiency = "🟢 Ottimo" if stats['utilization'] >= 90 else \
                    "🟡 Buono" if stats['utilization'] >= 75 else "🔴 Basso"
        
        print(f"Van {van_id:<6} {stats['loaded']:<12} {stats['total_time']:<15} "
              f"{stats['utilization']:<12.1f} {efficiency:<12}")
        total_time_all += stats['total_time']
    
    print("-" * 70)
    
    # Efficienza complessiva della flotta
    max_capacity = NUM_VANS * VAN_CAPACITY_MINUTES
    fleet_utilization = (total_time_all / max_capacity) * 100
    
    print(f"\n🚛 EFFICIENZA FLOTTA:")
    print(f"   • Capacità totale flotta: {max_capacity} minuti ({max_capacity/60:.0f} ore)")
    print(f"   • Tempo totale utilizzato: {total_time_all} minuti ({total_time_all/60:.1f} ore)")
    print(f"   • Utilizzo medio flotta: {fleet_utilization:.1f}%")
    
    # Raccomandazioni
    print(f"\n💡 RACCOMANDAZIONI:")
    if total_skipped > 0:
        print(f"   ⚠️  {total_skipped} pacchi richiedono un turno aggiuntivo o furgoni extra")
    if fleet_utilization < 80:
        print(f"   💰 Possibile riduzione flotta: utilizzo sotto l'80%")
    if fleet_utilization >= 95:
        print(f"   ✅ Flotta ottimizzata al massimo della capacità")


def save_results(df: pd.DataFrame, filepath: str):
    """
    Salva il DataFrame ottimizzato su file CSV.
    
    Args:
        df: DataFrame da salvare
        filepath: Percorso del file di output
    """
    print(f"\n💾 Salvataggio risultati...")
    
    # Ordinamento per leggibilità: prima per Van_ID, poi per Status
    df_sorted = df.sort_values(['Van_ID', 'Status', 'Time_Minutes'], 
                                ascending=[True, False, True])
    
    df_sorted.to_csv(filepath, index=False)
    print(f"✅ File salvato: {filepath}")
    
    # Preview delle nuove colonne
    print(f"\n📄 Preview colonne aggiunte:")
    print(df_sorted[['Predicted_Time', 'Time_Minutes', 'Van_ID', 'Status']].head(10).to_string())


# =============================================================================
# MAIN EXECUTION
# =============================================================================

def main():
    """
    Funzione principale che orchestra l'esecuzione dei 3 step.
    """
    print("\n" + "🚀" * 30)
    print("\n   AMAZON DELIVERY - VEHICLE ROUTING OPTIMIZATION")
    print("   Ricerca Operativa: Bin Packing & Clustering")
    print("\n" + "🚀" * 30)
    
    # Caricamento dati
    df = load_data(INPUT_FILE)
    
    # STEP 1: De-standardizzazione tempi
    df = destandardize_time(df)
    
    # STEP 2: Clustering geografico
    df = geographic_clustering(df)
    
    # STEP 3: Bin Packing Optimization
    df, van_stats = bin_packing_optimization(df)
    
    # Report finale
    print_final_report(df, van_stats)
    
    # Salvataggio risultati
    save_results(df, OUTPUT_FILE)
    
    print("\n" + "=" * 60)
    print("✅ OTTIMIZZAZIONE COMPLETATA CON SUCCESSO!")
    print("=" * 60 + "\n")
    
    return df

# Aggiungi questa cella PRIMA della cella main()

import sys
from io import StringIO
from datetime import datetime

class OutputCapture:
    """
    Classe per catturare l'output del terminale e salvarlo in Markdown.
    """
    def __init__(self):
        self.output = StringIO()
        self.original_stdout = sys.stdout
        
    def start(self):
        """Inizia la cattura dell'output."""
        sys.stdout = self.output
        
    def stop(self):
        """Ferma la cattura e ripristina stdout."""
        sys.stdout = self.original_stdout
        
    def get_output(self) -> str:
        """Restituisce l'output catturato."""
        return self.output.getvalue()
    
    def save_to_markdown(self, filepath: str = "csp_report.md"):
        """
        Salva l'output catturato in un file Markdown formattato.
        
        Args:
            filepath: Percorso del file di output
        """
        output_text = self.get_output()
        
        # Costruzione del documento Markdown
        markdown_content = f"""# 🚚 Vehicle Routing & Bin Packing - Report

**Data esecuzione:** {datetime.now().strftime("%d/%m/%Y %H:%M:%S")}

**Progetto:** Amazon Delivery AI - Ottimizzazione Flotta

---

## Output Completo

```
{output_text}
```

---

## Parametri Configurazione

| Parametro | Valore |
|-----------|--------|
| Numero Furgoni | {NUM_VANS} |
| Capacità per Furgone | {VAN_CAPACITY_MINUTES} minuti (8 ore) |
| Tempo Min Consegna | {MIN_DELIVERY_TIME} minuti |
| Tempo Max Consegna | {MAX_DELIVERY_TIME} minuti |

---

## Metodologia

### Step 1: De-Standardizzazione
Mapping lineare dei z-score → minuti reali [10, 120]

### Step 2: Clustering Geografico
K-Means con k={NUM_VANS} per divisione territoriale

### Step 3: Bin Packing Greedy
Ottimizzazione carico con priorità ai pacchi più veloci

---

*Report generato automaticamente dal sistema di ottimizzazione flotta*
"""
        
        # Salvataggio file
        with open(filepath, 'w', encoding='utf-8') as f:
            f.write(markdown_content)
        
        print(f"📝 Report Markdown salvato: {filepath}")


# Istanza globale per la cattura
capture = OutputCapture()

if __name__ == "__main__":
    # Esecuzione script
    optimized_df = main()

    capture.start()
    
    try:
        # Esecuzione principale
        optimized_df = main()
    finally:
        # Ferma cattura (anche in caso di errore)
        capture.stop()
    
    # Salva il report in Markdown
    capture.save_to_markdown("csp_report.md")
    
    # Ristampa l'output a schermo per visualizzarlo nel notebook
    print(capture.get_output())
    print("\n" + "=" * 60)
    print("📝 REPORT SALVATO IN: csp_report.md")
    print("=" * 60)


🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀

   AMAZON DELIVERY - VEHICLE ROUTING OPTIMIZATION
   Ricerca Operativa: Bin Packing & Clustering

🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀
CARICAMENTO DATI
✓ File caricato: ../Supervised_Learning/amazon_delivery_with_predictions.csv
✓ Numero totale consegne: 39997
✓ Colonne disponibili: ['Order_ID', 'Agent_Age', 'Agent_Rating', 'Store_Latitude', 'Store_Longitude', 'Drop_Latitude', 'Drop_Longitude', 'Delivery_Time', 'Weather_Cloudy', 'Weather_Fog', 'Weather_Sandstorms', 'Weather_Stormy', 'Weather_Sunny', 'Weather_Windy', 'Traffic_High', 'Traffic_Jam', 'Traffic_Low', 'Traffic_Medium', 'Vehicle_motorcycle', 'Vehicle_scooter', 'Vehicle_van', 'Area_Metropolitian', 'Area_Other', 'Area_Semi-Urban', 'Area_Urban', 'Category_Apparel', 'Category_Books', 'Category_Clothing', 'Category_Cosmetics', 'Category_Electronics', 'Category_Grocery', 'Category_Home', 'Category_Jewelry', 'Category_Kitchen', 'Category_Outdoors', 'Category_Pet Supplies', 'Category_Shoes', 'Category_Skinca

L'implementazione del solver Greedy Bin Packing combinato con il clustering K-Means ha prodotto risultati eccellenti in termini di saturazione delle risorse: ogni furgone viene utilizzato al 99.8% della sua capacità temporale (8 ore), massimizzando la produttività del singolo autista.

Tuttavia, l'analisi dei risultati evidenzia un gap dimensionale critico:

Con una capacità di circa 40-45 consegne giornaliere per veicolo, la flotta attuale di 5 mezzi copre meno dell'1% della domanda totale (40.000 ordini).

La distribuzione geografica non è uniforme: la "Zona 1" (probabile centro urbano identificato dal K-Means) presenta una densità di ordini 4 volte superiore alle zone periferiche, suggerendo la necessità di un'allocazione dinamica delle risorse (più furgoni nelle zone dense).

Raccomandazione Operativa: Il sistema è pronto per la produzione, ma richiede un ridimensionamento della flotta (scale-up) o una prioritizzazione delle consegne su più giorni lavorativi.

In [ ]:
"""
=============================================================================
FLEET CAPACITY PLANNING - ANALISI FABBISOGNO FLOTTA
=============================================================================
Progetto: Amazon Delivery AI
Modulo: Business Intelligence & Ricerca Operativa
Descrizione: Calcolo della flotta teorica ideale per coprire il 100% 
             degli ordini in un singolo turno lavorativo.

Formula chiave:
    Furgoni_Necessari = ⌈Tempo_Totale_Zona / Capacità_Turno⌉
    
Dove:
    - Tempo_Totale_Zona: Somma dei Time_Minutes per ogni cluster
    - Capacità_Turno: 480 minuti (8 ore lavorative)
    - ⌈x⌉: Funzione ceiling (arrotondamento per eccesso)
=============================================================================
"""

import pandas as pd
import numpy as np
from math import ceil
from datetime import datetime

# =============================================================================
# CONFIGURAZIONE PARAMETRI
# =============================================================================

# Parametri operativi
SHIFT_DURATION_MINUTES = 480    # Durata turno: 8 ore = 480 minuti
CURRENT_FLEET_SIZE = 5          # Flotta attualmente disponibile

# File I/O
INPUT_FILE = 'amazon_delivery_optimized.csv'
OUTPUT_REPORT = 'fleet_planning_report.md'


def load_data(filepath: str) -> pd.DataFrame:
    """
    Carica il dataset ottimizzato dal CSP.
    
    Args:
        filepath: Percorso del file CSV
        
    Returns:
        DataFrame con i dati delle consegne
    """
    print("=" * 70)
    print("CARICAMENTO DATI")
    print("=" * 70)
    
    df = pd.read_csv(filepath)
    
    print(f"✓ File caricato: {filepath}")
    print(f"✓ Record totali: {len(df):,}")
    print(f"✓ Colonne: {list(df.columns)}")
    
    return df


def calculate_zone_requirements(df: pd.DataFrame) -> pd.DataFrame:
    """
    Calcola il fabbisogno di furgoni per ogni zona geografica.
    
    Formula per ogni zona:
        Furgoni = ⌈Σ(Time_Minutes) / 480⌉
    
    Args:
        df: DataFrame con colonne 'Van_ID' e 'Time_Minutes'
        
    Returns:
        DataFrame aggregato con metriche per zona
    """
    print("\n" + "=" * 70)
    print("ANALISI FABBISOGNO PER ZONA (CAPACITY PLANNING)")
    print("=" * 70)
    
    # Aggregazione per zona
    zone_stats = df.groupby('Van_ID').agg(
        Total_Orders=('Time_Minutes', 'count'),
        Total_Time_Minutes=('Time_Minutes', 'sum'),
        Avg_Delivery_Time=('Time_Minutes', 'mean'),
        Min_Delivery_Time=('Time_Minutes', 'min'),
        Max_Delivery_Time=('Time_Minutes', 'max')
    ).reset_index()
    
    # Calcolo furgoni necessari per zona
    # Formula: ceil(Tempo_Totale / Capacità_Turno)
    zone_stats['Vans_Required'] = zone_stats['Total_Time_Minutes'].apply(
        lambda x: ceil(x / SHIFT_DURATION_MINUTES)
    )
    
    # Calcolo ore totali per zona (più leggibile)
    zone_stats['Total_Hours'] = zone_stats['Total_Time_Minutes'] / 60
    
    # Calcolo saturazione teorica dei furgoni assegnati
    # Saturazione = (Tempo_Totale / (Furgoni * Capacità)) * 100
    zone_stats['Fleet_Saturation_%'] = (
        zone_stats['Total_Time_Minutes'] / 
        (zone_stats['Vans_Required'] * SHIFT_DURATION_MINUTES) * 100
    )
    
    # Capacità residua (minuti liberi)
    zone_stats['Spare_Capacity_Min'] = (
        zone_stats['Vans_Required'] * SHIFT_DURATION_MINUTES - 
        zone_stats['Total_Time_Minutes']
    )
    
    return zone_stats


def calculate_fleet_totals(zone_stats: pd.DataFrame) -> dict:
    """
    Calcola le metriche aggregate per l'intera flotta.
    
    Args:
        zone_stats: DataFrame con statistiche per zona
        
    Returns:
        Dizionario con metriche globali
    """
    totals = {
        'total_orders': zone_stats['Total_Orders'].sum(),
        'total_time_minutes': zone_stats['Total_Time_Minutes'].sum(),
        'total_time_hours': zone_stats['Total_Hours'].sum(),
        'total_vans_required': zone_stats['Vans_Required'].sum(),
        'current_fleet': CURRENT_FLEET_SIZE,
        'fleet_deficit': zone_stats['Vans_Required'].sum() - CURRENT_FLEET_SIZE,
        'avg_saturation': zone_stats['Fleet_Saturation_%'].mean(),
        'total_spare_capacity': zone_stats['Spare_Capacity_Min'].sum()
    }
    
    # Calcolo copertura con flotta attuale
    current_capacity = CURRENT_FLEET_SIZE * SHIFT_DURATION_MINUTES
    totals['current_coverage_%'] = min(100, (current_capacity / totals['total_time_minutes']) * 100)
    
    # Giorni necessari con flotta attuale
    totals['days_required_current_fleet'] = ceil(totals['total_time_minutes'] / current_capacity)
    
    return totals


def print_zone_table(zone_stats: pd.DataFrame):
    """
    Stampa una tabella ASCII formattata con le statistiche per zona.
    
    Args:
        zone_stats: DataFrame con statistiche per zona
    """
    print("\n" + "=" * 90)
    print("📊 RIEPILOGO FABBISOGNO PER ZONA GEOGRAFICA")
    print("=" * 90)
    
    # Header
    print(f"{'Zona':<8} {'Ordini':<10} {'Tempo Tot':<12} {'Ore Tot':<10} "
          f"{'Furgoni':<10} {'Saturaz.':<12} {'Capacità':<10}")
    print(f"{'(ID)':<8} {'(n°)':<10} {'(min)':<12} {'(h)':<10} "
          f"{'Necessari':<10} {'(%)':<12} {'Residua':<10}")
    print("-" * 90)
    
    # Righe dati
    for _, row in zone_stats.iterrows():
        print(f"Zona {int(row['Van_ID']):<3} {int(row['Total_Orders']):<10,} "
              f"{int(row['Total_Time_Minutes']):<12,} {row['Total_Hours']:<10.1f} "
              f"{int(row['Vans_Required']):<10} {row['Fleet_Saturation_%']:<12.1f} "
              f"{int(row['Spare_Capacity_Min']):<10} min")
    
    print("-" * 90)
    
    # Totali
    print(f"{'TOTALE':<8} {int(zone_stats['Total_Orders'].sum()):<10,} "
          f"{int(zone_stats['Total_Time_Minutes'].sum()):<12,} "
          f"{zone_stats['Total_Hours'].sum():<10.1f} "
          f"{int(zone_stats['Vans_Required'].sum()):<10} "
          f"{zone_stats['Fleet_Saturation_%'].mean():<12.1f} "
          f"{int(zone_stats['Spare_Capacity_Min'].sum()):<10} min")
    print("=" * 90)


def print_gap_analysis(totals: dict):
    """
    Stampa l'analisi del gap tra flotta attuale e fabbisogno.
    
    Args:
        totals: Dizionario con metriche globali
    """
    print("\n" + "=" * 70)
    print("🔍 GAP ANALYSIS - CONFRONTO FLOTTA ATTUALE vs FABBISOGNO")
    print("=" * 70)
    
    print(f"""
┌─────────────────────────────────────────────────────────────────────┐
│                    ANALISI CAPACITÀ FLOTTA                         │
├─────────────────────────────────────────────────────────────────────┤
│  📦 Ordini Totali da Evadere:        {totals['total_orders']:>10,} ordini           │
│  ⏱️  Tempo Totale Richiesto:          {totals['total_time_minutes']:>10,} minuti          │
│                                      ({totals['total_time_hours']:>10.1f} ore)            │
├─────────────────────────────────────────────────────────────────────┤
│  🚚 Flotta Attuale:                  {totals['current_fleet']:>10} furgoni          │
│  🚛 Flotta Necessaria (100%):        {totals['total_vans_required']:>10} furgoni          │
│  ⚠️  DEFICIT FURGONI:                 {totals['fleet_deficit']:>10} furgoni          │
├─────────────────────────────────────────────────────────────────────┤
│  📈 Copertura con Flotta Attuale:    {totals['current_coverage_%']:>10.2f}%              │
│  📅 Giorni per Evadere (5 furgoni):  {totals['days_required_current_fleet']:>10} giorni           │
│  📊 Saturazione Media Flotta Ideale: {totals['avg_saturation']:>10.1f}%              │
└─────────────────────────────────────────────────────────────────────┘
""")


def print_management_conclusions(totals: dict, zone_stats: pd.DataFrame):
    """
    Stampa le conclusioni manageriali e raccomandazioni strategiche.
    
    Args:
        totals: Dizionario con metriche globali
        zone_stats: DataFrame con statistiche per zona
    """
    print("\n" + "=" * 70)
    print("💼 CONCLUSIONI MANAGERIALI & RACCOMANDAZIONI STRATEGICHE")
    print("=" * 70)
    
    # Identificazione zona più critica
    critical_zone = zone_stats.loc[zone_stats['Vans_Required'].idxmax()]
    
    deficit_percentage = (totals['fleet_deficit'] / totals['total_vans_required']) * 100
    
    print(f"""
╔═══════════════════════════════════════════════════════════════════════╗
║                      EXECUTIVE SUMMARY                                ║
╠═══════════════════════════════════════════════════════════════════════╣
║                                                                       ║
║  🔴 CRITICITÀ PRINCIPALE:                                             ║
║     La flotta attuale di {totals['current_fleet']} furgoni copre solo il {totals['current_coverage_%']:.1f}% della       ║
║     domanda giornaliera. Sono necessari {totals['total_vans_required']} furgoni per garantire     ║
║     il 100% delle consegne in un singolo turno.                       ║
║                                                                       ║
║  📊 DIMENSIONAMENTO GAP:                                              ║
║     • Deficit: {totals['fleet_deficit']} furgoni ({deficit_percentage:.1f}% della flotta ideale)             ║
║     • Zona più critica: Zona {int(critical_zone['Van_ID'])} ({int(critical_zone['Vans_Required'])} furgoni necessari)         ║
║     • Con la flotta attuale servono {totals['days_required_current_fleet']} giorni lavorativi           ║
║                                                                       ║
╠═══════════════════════════════════════════════════════════════════════╣

""")


def save_report_markdown(zone_stats: pd.DataFrame, totals: dict, filepath: str):
    """
    Salva il report completo in formato Markdown.
    
    Args:
        zone_stats: DataFrame con statistiche per zona
        totals: Dizionario con metriche globali
        filepath: Percorso file di output
    """
    
    # Costruzione tabella zone per Markdown
    zone_table = "| Zona | Ordini | Tempo (min) | Ore | Furgoni | Saturazione |\n"
    zone_table += "|------|--------|-------------|-----|---------|-------------|\n"
    
    for _, row in zone_stats.iterrows():
        zone_table += f"| {int(row['Van_ID'])} | {int(row['Total_Orders']):,} | "
        zone_table += f"{int(row['Total_Time_Minutes']):,} | {row['Total_Hours']:.1f} | "
        zone_table += f"{int(row['Vans_Required'])} | {row['Fleet_Saturation_%']:.1f}% |\n"
    
    zone_table += f"| **TOTALE** | **{int(totals['total_orders']):,}** | "
    zone_table += f"**{int(totals['total_time_minutes']):,}** | **{totals['total_time_hours']:.1f}** | "
    zone_table += f"**{int(totals['total_vans_required'])}** | **{totals['avg_saturation']:.1f}%** |\n"
    
    markdown_content = f"""# 🚛 Fleet Capacity Planning Report

**Data Analisi:** {datetime.now().strftime("%d/%m/%Y %H:%M:%S")}

**Progetto:** Amazon Delivery AI - Business Intelligence

---

## 📊 Executive Summary

| Metrica | Valore |
|---------|--------|
| Ordini Totali | {int(totals['total_orders']):,} |
| Tempo Totale Richiesto | {int(totals['total_time_minutes']):,} minuti ({totals['total_time_hours']:.1f} ore) |
| Flotta Attuale | {totals['current_fleet']} furgoni |
| **Flotta Necessaria** | **{totals['total_vans_required']} furgoni** |
| **Deficit** | **{totals['fleet_deficit']} furgoni** |
| Copertura Attuale | {totals['current_coverage_%']:.2f}% |

---

## 📈 Analisi per Zona Geografica

{zone_table}

---

## 🔍 Gap Analysis

- **Deficit Furgoni:** {totals['fleet_deficit']} unità
- **Percentuale Deficit:** {(totals['fleet_deficit'] / totals['total_vans_required']) * 100:.1f}%
- **Giorni necessari (flotta attuale):** {totals['days_required_current_fleet']}

---

## 💼 Raccomandazioni Strategiche

### Opzione A: Scale-Up Flotta
- Leasing operativo di {totals['fleet_deficit']} furgoni
- Costo stimato: €{totals['fleet_deficit'] * 1500:,}/mese

### Opzione B: Doppio Turno
- Furgoni per turno: {ceil(totals['total_vans_required']/2)}
- Richiede assunzione autisti aggiuntivi

### Opzione C: Outsourcing Last-Mile
- Partnership con corrieri terzi
- Costo variabile: €2-4/consegna

---

## ✅ Azione Consigliata

**Combinazione Opzione A + C:**
1. Leasing immediato di {min(totals['fleet_deficit'], 50)} furgoni
2. Partnership last-mile per picchi di domanda
3. Implementazione graduale doppio turno (Q2 2026)

---

*Report generato automaticamente dal sistema Fleet Planning*
"""
    
    with open(filepath, 'w', encoding='utf-8') as f:
        f.write(markdown_content)
    
    print(f"\n💾 Report Markdown salvato: {filepath}")


def main():
    """
    Funzione principale - Orchestra l'analisi del fabbisogno flotta.
    """
    print("\n" + "🚛" * 35)
    print("\n   FLEET CAPACITY PLANNING - ANALISI FABBISOGNO FLOTTA")
    print("   Business Intelligence & Ricerca Operativa")
    print("\n" + "🚛" * 35)
    
    # 1. Caricamento dati
    df = load_data(INPUT_FILE)
    
    # 2. Calcolo fabbisogno per zona
    zone_stats = calculate_zone_requirements(df)
    
    # 3. Calcolo totali flotta
    totals = calculate_fleet_totals(zone_stats)
    
    # 4. Output tabellare
    print_zone_table(zone_stats)
    
    # 5. Gap Analysis
    print_gap_analysis(totals)
    
    # 6. Conclusioni manageriali
    print_management_conclusions(totals, zone_stats)
    
    # 7. Salvataggio report
    save_report_markdown(zone_stats, totals, OUTPUT_REPORT)
    
    print("\n" + "=" * 70)
    print("✅ ANALISI FLEET PLANNING COMPLETATA!")
    print("=" * 70 + "\n")
    
    return zone_stats, totals


if __name__ == "__main__":
    zone_stats, totals = main()


🚛🚛🚛🚛🚛🚛🚛🚛🚛🚛🚛🚛🚛🚛🚛🚛🚛🚛🚛🚛🚛🚛🚛🚛🚛🚛🚛🚛🚛🚛🚛🚛🚛🚛🚛

   FLEET CAPACITY PLANNING - ANALISI FABBISOGNO FLOTTA
   Business Intelligence & Ricerca Operativa

🚛🚛🚛🚛🚛🚛🚛🚛🚛🚛🚛🚛🚛🚛🚛🚛🚛🚛🚛🚛🚛🚛🚛🚛🚛🚛🚛🚛🚛🚛🚛🚛🚛🚛🚛
CARICAMENTO DATI
✓ File caricato: amazon_delivery_optimized.csv
✓ Record totali: 39,997
✓ Colonne: ['Order_ID', 'Agent_Age', 'Agent_Rating', 'Store_Latitude', 'Store_Longitude', 'Drop_Latitude', 'Drop_Longitude', 'Delivery_Time', 'Weather_Cloudy', 'Weather_Fog', 'Weather_Sandstorms', 'Weather_Stormy', 'Weather_Sunny', 'Weather_Windy', 'Traffic_High', 'Traffic_Jam', 'Traffic_Low', 'Traffic_Medium', 'Vehicle_motorcycle', 'Vehicle_scooter', 'Vehicle_van', 'Area_Metropolitian', 'Area_Other', 'Area_Semi-Urban', 'Area_Urban', 'Category_Apparel', 'Category_Books', 'Category_Clothing', 'Category_Cosmetics', 'Category_Electronics', 'Category_Grocery', 'Category_Home', 'Category_Jewelry', 'Category_Kitchen', 'Category_Outdoors', 'Category_Pet Supplies', 'Category_Shoes', 'Category_Skincare', 'Category_Snacks', 'Category_Spo